# Figures Calcul 001

In [10]:
# import necessary libraries
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import re
from typing import Dict, List, Tuple, Any
from matplotlib.patches import Circle

In [11]:
# File paths
data_path = "../data/001MoDe_R1.csv"
marker_path = "../data/001MoDe_R1.marker.csv"

### 1. Parse Header From Data File

In [12]:
def parse_header(data_path: str) -> Dict[str, Any]:
    """Parses the header from the first non-empty line of the data file.
    
    - Objective:
            Extract key–value metadata stored in the first non-empty line of a CSV-like
            file. The function handles multiple formats such as `key=value`,
            `key:value`, or `key value`.

    - Inputs:
            data_path (str): Path to the data file from which the header should be read.

    - Outputs:
            Dict[str, Any]: A dictionary containing the parsed header fields.  
            Numeric values are converted to float when possible; all other values  
            remain as strings. Returns an empty dictionary if the file is not found  
             or if no header line is detected.
    """
    header: Dict[str, Any] = {}
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            first_line = ""
            for line in f:
                stripped_line = line.strip()
                if stripped_line:
                    first_line = stripped_line
                    break

            if not first_line:
                return header

            for part in first_line.split(";"):
                part = part.strip()
                if not part:
                    continue

                if "=" in part:
                    k, v = part.split("=", 1)
                elif ":" in part:
                    k, v = part.split(":", 1)
                else:
                    tokens = part.split()
                    k = tokens[0]
                    v = tokens[1] if len(tokens) > 1 else ""

                k, v = k.strip(), v.strip()
                try:
                    header[k] = float(v)
                except (ValueError, TypeError):
                    header[k] = v

    except FileNotFoundError:
        print(f"Error: Data file not found at {data_path}")
        return {}

    return header

### 2. Parse Markers From Marker File

In [13]:
def parse_markers(marker_path: str) -> Dict[str, Any]:
    """Parse a marker file to extract recording start and end timestamps.

    - Objective:
            Read a marker log file and identify the timestamps corresponding to 
            "DoRecord" and "DoPause" events. These markers define the start and end 
            of recorded segments. The function also provides the full list of markers 
            and matches each start event to its corresponding end event.

    - Inputs:
            marker_path (str): Path to the marker file. The file is expected to contain
            lines formatted with at least two comma-separated fields, where the
            second field is a timestamp.

    - Outputs:
            Dict[str, Any]: A dictionary containing:
                "start_ts" (List[int]): Sorted list of detected start timestamps.
                "end_ts" (List[int]): Sorted list of detected end timestamps.
                "pairs" (List[Tuple[int, int]]): List of matched (start, end) timestamp pairs.
                "all_markers" (List[Tuple[int, str]]): All markers found, each as (timestamp, raw_line).
      
      Raises informative exceptions if the marker file is missing or if no valid
      start/end markers can be extracted.
    """
    record_starts_raw: List[int] = []
    record_ends_raw: List[int] = []
    all_markers: List[Tuple[int, str]] = []

    try:
        with open(marker_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        raise FileNotFoundError(f"Marker file not found at {marker_path}")

    for line in lines:
        parts = line.strip().split(",")
        if len(parts) < 2:
            continue

        ts_str = parts[1].strip().replace(" ", "")
        try:
            ts = int(ts_str)
        except ValueError:
            continue

        all_markers.append((ts, line.strip()))

        if "DoCycleChange:DoRecord" in line:
            record_starts_raw.append(ts)
        elif "DoCycleChange:DoPause" in line:
            record_ends_raw.append(ts)

    if not record_starts_raw:
        raise ValueError("No 'DoRecord' events detected in the marker file.")
    if not record_ends_raw:
        raise ValueError("No 'DoPause' events detected in the marker file.")

    record_starts_raw.sort()
    record_ends_raw.sort()

    pairs: List[Tuple[int, int]] = []
    end_idx = 0
    for start_ts in record_starts_raw:
        while end_idx < len(record_ends_raw) and record_ends_raw[end_idx] < start_ts:
            end_idx += 1
        if end_idx < len(record_ends_raw):
            pairs.append((start_ts, record_ends_raw[end_idx]))
            end_idx += 1
        else:
            break

    if not pairs:
        raise RuntimeError("No valid start/end pairs found in markers.")

    return {
        "start_ts": [p[0] for p in pairs],
        "end_ts": [p[1] for p in pairs],
        "pairs": pairs,
        "all_markers": all_markers,
    }

### 3. Load Time-Series Data From CSV File

In [ ]:
def load_data(data_path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load timestamp, mouse position (x, y), and target indicator data from a CSV file.
    
    - Inputs:
            data_path (str): Path to the CSV file.

    - Outputs:
            Tuple of four np.ndarray:
                timestamps, mouse_x, mouse_y, in_target.
                Returns empty arrays if the file is not found."""
    
    timestamps: List[float] = []
    mouse_x: List[float] = []
    mouse_y: List[float] = []
    in_target: List[int] = []

    try:
        with open(data_path, "r", encoding="utf-8") as f:
            # Skip header lines until a data row is found
            for line in f:
                if (
                    line.strip()
                    and re.match(r"^-?\d+", line.strip().split(";")[0].split(",")[0])
                ):
                    parts = line.strip().split(";")[0].split(",")
                    if len(parts) >= 4:
                        try:
                            timestamps.append(float(parts[0]))
                            mouse_x.append(float(parts[1]))
                            mouse_y.append(float(parts[2]))
                            in_target.append(int(float(parts[3])))
                        except ValueError:
                            pass
                    break

            # Continue reading the rest of the file
            reader = csv.reader(f, delimiter=";")
            for row in reader:
                if len(row) >= 1 and "," in row[0]:
                    parts = row[0].split(",")
                    if len(parts) >= 4:
                        try:
                            timestamps.append(float(parts[0]))
                            mouse_x.append(float(parts[1]))
                            mouse_y.append(float(parts[2]))
                            in_target.append(int(float(parts[3])))
                        except ValueError:
                            continue

    except FileNotFoundError:
        print(f"Error: Data file not found at {data_path}")
        return np.array([]), np.array([]), np.array([]), np.array([])

    return (
        np.array(timestamps),
        np.array(mouse_x),
        np.array(mouse_y),
        np.array(in_target),
    )

### 4. Plot Overall Time Series

In [14]:
def plot_overall_timeseries(
    time: np.ndarray,
    mouse_x: np.ndarray,
    mouse_y: np.ndarray,
    record_starts_sec: List[float],
    record_ends_sec: List[float],
) -> None:
    """Plot centered mouse x/y time series and highlight valid recording intervals.

    - Inputs:
            time (np.ndarray): Time vector.
            mouse_x, mouse_y (np.ndarray): Raw cursor coordinates.
            record_starts_sec, record_ends_sec (List[float]): Start/end times of valid segments.

    - Outputs:
            None. Displays a plot of x and y positions with masked invalid periods."""

    # Center the data
    plot_x = mouse_x - np.mean(mouse_x)
    plot_y = mouse_y - np.mean(mouse_y)

    # Mask valid intervals
    valid_mask = np.zeros_like(time, dtype=bool)
    for start, end in zip(record_starts_sec, record_ends_sec):
        valid_mask |= (time >= start) & (time <= end)

    plot_x_masked = np.copy(plot_x)
    plot_y_masked = np.copy(plot_y)
    plot_x_masked[~valid_mask] = np.nan
    plot_y_masked[~valid_mask] = np.nan

    plt.figure(figsize=(15, 6))
    plt.plot(time, plot_x_masked, ".", ms=4, color="tab:blue", label="x position")
    plt.plot(time, plot_y_masked, ".", ms=4, color="tab:orange", label="y position")

    for i, s in enumerate(record_starts_sec):
        plt.axvline(
            s,
            color="black",
            linestyle="--",
            linewidth=1.5,
            label="record start" if i == 0 else "",
        )

    for i, e in enumerate(record_ends_sec):
        plt.axvline(
            e,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label="record end" if i == 0 else "",
        )

    plt.xlim(
        min(time[0], record_starts_sec[0]) - 1,
        max(time[-1], record_ends_sec[-1]) + 1,
    )
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title("Raw time series data")
    plt.legend()
    plt.grid(False)
    plt.tight_layout()
    plt.show()

### 5. Plot Single Record Trajectory

In [15]:
def plot_record_trajectory(
    record_idx: int,
    data: Dict[str, np.ndarray],
    header: Dict[str, Any],
) -> None:
    """Plot the 2D cursor trajectory for a selected recording, with points
    inside/outside the target and screen borders.

    - Inputs:
            record_idx (int): Index of the recording to display.
            data (dict): Contains "x", "y", and "in_target" arrays.
            header (dict): Must provide screenWidth and screenHeight.

    - Outputs:
            None. Displays the trajectory with target ring and screen frame.
    """

    fig, ax = plt.subplots(figsize=(8, 5))

    x_task, y_task, in_task = data["x"], data["y"], data["in_target"]

    # Trajectory + points
    ax.plot(x_task, y_task, "-", color="steelblue", zorder=2, lw=0.5)
    ax.scatter(
        x_task[in_task == 1],
        y_task[in_task == 1],
        color="green",
        s=10,
        zorder=3,
        lw=0.3,
    )
    ax.scatter(
        x_task[in_task == 0],
        y_task[in_task == 0],
        color="red",
        s=10,
        zorder=3,
        lw=0.8,
    )

    # Distance of each point from the center (0,0)
    dist = np.sqrt(x_task**2 + y_task**2)

    # Keep only the "inside" points
    dist_inside = dist[in_task == 1]

    if dist_inside.size > 0:
        # Inner / outer radius derived from inside points
        inner_r = dist_inside.min()
        outer_r = dist_inside.max()

        circle_outer = plt.Circle(
            (0, 0),
            outer_r,
            color="yellow",
            alpha=0.5,
            zorder=1,
        )
        circle_inner = plt.Circle(
            (0, 0),
            inner_r,
            color="white",
            zorder=2,
        )

        ax.add_patch(circle_outer)
        ax.add_patch(circle_inner)

    # Screen border (as before)
    sw, sh = header.get("screenWidth", 0), header.get("screenHeight", 0)
    ax.plot(
        [-sw / 2, sw / 2, sw / 2, -sw / 2, -sw / 2],
        [-sh / 2, -sh / 2, sh / 2, sh / 2, -sh / 2],
        color="magenta",
        lw=1.2,
        zorder=0,
    )

    # Custom legend
    screen_leg = Patch(
        facecolor="none",
        edgecolor="magenta",
        linewidth=1.2,
        label="screen",
    )
    # Target in legend: solid yellow
    target_leg = Patch(
        facecolor="yellow",
        edgecolor="yellow",
        linewidth=0,
        label="target",
    )
    inside_leg = Line2D(
        [0],
        [0],
        marker="o",
        color="green",
        linestyle="None",
        markersize=4,
        label="inside",
    )
    outside_leg = Line2D(
        [0],
        [0],
        marker="o",
        color="red",
        linestyle="None",
        markersize=4,
        label="outside",
    )

    ax.legend(
        handles=[screen_leg, target_leg, inside_leg, outside_leg],
        loc="upper right",
    )

    ax.set_title(f"Record {record_idx}")
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Y (pixels)")
    ax.axis("equal")
    ax.grid(False)

    plt.show()


### 6. Plot X, Y and In-Target Time Series for One Record

In [16]:
def plot_record_temporal(record_idx: int, data: Dict[str, np.ndarray]) -> None:
    """Plot the temporal evolution of x position, y position, and in-target flag
    for a selected recording

    - Inputs:
            record_idx (int): Index of the recording to plot.
            data (dict): Must contain "time", "x", "y", and "in_target" arrays.

    - Outputs:
            None. Displays three time-series plots (x, y, is_inside)."""
            
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    fig.suptitle(f"Record {record_idx}")

    t_task, x_task, y_task, in_task = (
        data["time"],
        data["x"],
        data["y"],
        data["in_target"],
    )

    # X Position
    axes[0].plot(
        t_task,
        x_task,
        linestyle="-",
        marker="o",
        color="blue",
        label="x",
        lw=0.5,
        markersize=2,
    )
    axes[0].set_ylabel("X (pixels)")
    axes[0].legend()
    axes[0].grid(True)

    # Y Position
    axes[1].plot(
        t_task,
        y_task,
        linestyle="-",
        marker="o",
        color="orange",
        label="y",
        lw=0.5,
        markersize=2,
    )
    axes[1].set_ylabel("Y (pixels)")
    axes[1].legend()
    axes[1].grid(True)

    # In Target Flag (step)
    axes[2].step(
        t_task,
        in_task,
        linestyle="-",
        marker="o",
        where="post",
        color="green",
        label="is_inside",
        lw=0.5,
        markersize=2,
    )
    axes[2].set_ylabel("Is inside")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()